# Person 5 — Final training and held-out evaluation

Part of the six-person Basil Leaf ML pipeline. Run the numbered notebooks in order. This notebook states its inputs, produces a concrete handoff in `parts/artifacts`, and does not overwrite the complete project's `outputs/` results.

## Responsibility
Recreate the selected estimator, train it once on all development images, evaluate it once on the fixed held-out test set, calculate class metrics and a group-bootstrap interval, and save the model.

**Input:** Persons 2–4 artifacts  
**Output:** `05_model.joblib`, `05_test_predictions.csv`, and `05_evaluation.json`

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "Basil_Leaf_ML_Workflow.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project folder or parts folder.")
DATA_DIR = ROOT / "data" / "raw"
ARTIFACTS = ROOT / "parts" / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SEED = 42
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
FOLDERS = {
    "Amravati_Region_Basil_Plant_Healthy": ("Healthy", 31),
    "Nagpur_Region_Basil_Plant_Healthy": ("Healthy", 473),
    "Pune_Region_Basil_Plant_Healthy": ("Healthy", 146),
    "Basil_Plant_Unhealthy": ("Unhealthy", 481),
}
print("Project:", ROOT)
print("Python:", sys.executable)
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score

Project: D:\SLIIT\projectr\Dataset_Train
Python: D:\SLIIT\projectr\Dataset_Train\.venv\Scripts\python.exe


In [2]:
manifest=pd.read_csv(ARTIFACTS/'02_clean_manifest.csv'); X=np.load(ARTIFACTS/'03_features.npz')['X']; y=manifest.label.to_numpy()
selection=json.loads((ARTIFACTS/'04_selection.json').read_text())
if selection['selected_model']!='Random forest': raise ValueError('This notebook currently defines the validated Random forest winner; add the selected estimator configuration if the winner changes.')
dev=np.flatnonzero(manifest.split.eq('development')); test=np.flatnonzero(manifest.split.eq('test'))
model=RandomForestClassifier(n_estimators=200,min_samples_leaf=2,max_features='sqrt',class_weight='balanced',random_state=SEED,n_jobs=2)
model.fit(X[dev],y[dev]); pred=model.predict(X[test]); classes=list(model.classes_)
metrics={'accuracy':float(accuracy_score(y[test],pred)),'balanced_accuracy':float(balanced_accuracy_score(y[test],pred)),'macro_f1':float(f1_score(y[test],pred,average='macro')),'development_images':len(dev),'test_images':len(test),'classes':classes,'confusion_matrix':confusion_matrix(y[test],pred,labels=classes).tolist(),'classification_report':classification_report(y[test],pred,labels=classes,output_dict=True,zero_division=0)}
rng=np.random.default_rng(SEED); test_groups=manifest.split_group.to_numpy()[test]; unique=np.unique(test_groups); boot=[]
for _ in range(500):
    sampled=rng.choice(unique,len(unique),replace=True); idx=np.concatenate([np.flatnonzero(test_groups==g) for g in sampled])
    if set(y[test][idx])==set(classes): boot.append(f1_score(y[test][idx],pred[idx],average='macro'))
metrics['macro_f1_group_bootstrap_95_interval']=np.quantile(boot,[.025,.975]).tolist()
bundle={'estimator':model,'feature':'handcrafted','feature_version':'rgb-hsv-lbp-hog-v1','name':'Random forest','classes':classes,'seed':SEED}
joblib.dump(bundle,ARTIFACTS/'05_model.joblib')
out=manifest.iloc[test][['path','label','split_group']].copy(); out['prediction']=pred; out.to_csv(ARTIFACTS/'05_test_predictions.csv',index=False)
(ARTIFACTS/'05_evaluation.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
display(pd.DataFrame(metrics['classification_report']).T); print({k:v for k,v in metrics.items() if k not in {'classification_report'}})

,precision,recall,f1-score,support
Healthy,0.920000,1.000000,0.958333,92.000000
Unhealthy,1.000000,0.909091,0.952381,88.000000
accuracy,0.955556,0.955556,0.955556,0.955556
macro avg,0.960000,0.954545,0.955357,180.000000
weighted avg,0.959111,0.955556,0.955423,180.000000


{'accuracy': 0.9555555555555556, 'balanced_accuracy': 0.9545454545454546, 'macro_f1': 0.9553571428571428, 'development_images': 717, 'test_images': 180, 'classes': ['Healthy', 'Unhealthy'], 'confusion_matrix': [[92, 0], [8, 80]], 'macro_f1_group_bootstrap_95_interval': [0.9220655008138102, 0.9832552925047623]}


## Handoff to Person 6
Provide the trusted model, evaluation report, and test predictions. Highlight errors—especially unhealthy leaves predicted as healthy—and the limits of a partial dataset.